In [ ]:
#Cell 1 - Project configuration and path setup
from pathlib import Path

# Main thesis root on Google Drive
THESIS_ROOT = Path('/content/drive/MyDrive/NyanThesisAnomaly')

# Project folders
DATA_RAW_DIR = THESIS_ROOT / 'data' / 'raw'
DATA_INTERIM_DIR = THESIS_ROOT / 'data' / 'interim'
DATA_PROCESSED_DIR = THESIS_ROOT / 'data' / 'processed'
OUTPUTS_DIR = THESIS_ROOT / 'outputs'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
TABLES_DIR = OUTPUTS_DIR / 'tables'
FILES_DIR = OUTPUTS_DIR / 'files'
LOGS_DIR = OUTPUTS_DIR / 'logs'
NOTEBOOK_EXPORTS_DIR = OUTPUTS_DIR / 'notebook_exports'

PHASE1_DIR = OUTPUTS_DIR / 'phase1_benchmark'
PHASE2_DIR = OUTPUTS_DIR / 'phase2_tuning'
PHASE3_DIR = OUTPUTS_DIR / 'phase3_expert_review'

print('Configured thesis root:', THESIS_ROOT)


Configured thesis root: /content/drive/MyDrive/NyanThesisAnomaly


In [ ]:
#Cell 2 - Mount Google Drive
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
#Cell 3 - Create thesis folder structure in Google Drive
for folder in [
    DATA_RAW_DIR,
    DATA_INTERIM_DIR,
    DATA_PROCESSED_DIR,
    OUTPUTS_DIR,
    FIGURES_DIR,
    TABLES_DIR,
    FILES_DIR,
    LOGS_DIR,
    NOTEBOOK_EXPORTS_DIR,
    PHASE1_DIR,
    PHASE2_DIR,
    PHASE3_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print('Created/verified project folders:')
for folder in [DATA_RAW_DIR, DATA_INTERIM_DIR, DATA_PROCESSED_DIR, PHASE1_DIR, PHASE2_DIR, PHASE3_DIR]:
    print('-', folder)


Created/verified project folders:
- /content/drive/MyDrive/NyanThesisAnomaly/data/raw
- /content/drive/MyDrive/NyanThesisAnomaly/data/interim
- /content/drive/MyDrive/NyanThesisAnomaly/data/processed
- /content/drive/MyDrive/NyanThesisAnomaly/outputs/phase1_benchmark
- /content/drive/MyDrive/NyanThesisAnomaly/outputs/phase2_tuning
- /content/drive/MyDrive/NyanThesisAnomaly/outputs/phase3_expert_review


In [ ]:
#Cell 4 - Install required libraries for Colab
!pip -q install kagglehub pyarrow pandas numpy scikit-learn matplotlib seaborn openpyxl


In [ ]:
#Cell 5 - Download outpatient claims dataset from Kaggle
import kagglehub

# Download latest version
kaggle_path = kagglehub.dataset_download('kukulauren/cms-desynpuf-2010-outpatient-claims')

print('Path to dataset files:', kaggle_path)


100%|██████████| 9.15M/9.15M [00:00<00:00, 51.3MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/kukulauren/cms-desynpuf-2010-outpatient-claims/versions/1


In [ ]:
#Cell 6 - Inspect downloaded dataset files
from pathlib import Path

kaggle_dir = Path(kaggle_path)
downloaded_files = sorted([str(p) for p in kaggle_dir.rglob('*') if p.is_file()])

print('Number of downloaded files:', len(downloaded_files))
for f in downloaded_files[:50]:
    print(f)


Number of downloaded files: 1
/root/.cache/kagglehub/datasets/kukulauren/cms-desynpuf-2010-outpatient-claims/versions/1/desynpuf_outpatient_claims_2010_filtered.csv


In [ ]:
#Cell 7 - Copy the downloaded dataset into Google Drive raw-data storage
import shutil

KAGGLE_RAW_COPY_DIR = DATA_RAW_DIR / 'kaggle_cms_desynpuf_2010_outpatient_claims'
KAGGLE_RAW_COPY_DIR.mkdir(parents=True, exist_ok=True)

copied = 0
for src in kaggle_dir.rglob('*'):
    if src.is_file():
        rel = src.relative_to(kaggle_dir)
        dst = KAGGLE_RAW_COPY_DIR / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if not dst.exists():
            shutil.copy2(src, dst)
            copied += 1

print('Raw dataset copy location:', KAGGLE_RAW_COPY_DIR)
print('Newly copied files:', copied)


Raw dataset copy location: /content/drive/MyDrive/NyanThesisAnomaly/data/raw/kaggle_cms_desynpuf_2010_outpatient_claims
Newly copied files: 1


In [ ]:
#Cell 8 - Register reusable project paths for later notebooks
import json

project_paths = {
    'THESIS_ROOT': str(THESIS_ROOT),
    'DATA_RAW_DIR': str(DATA_RAW_DIR),
    'DATA_INTERIM_DIR': str(DATA_INTERIM_DIR),
    'DATA_PROCESSED_DIR': str(DATA_PROCESSED_DIR),
    'OUTPUTS_DIR': str(OUTPUTS_DIR),
    'FIGURES_DIR': str(FIGURES_DIR),
    'TABLES_DIR': str(TABLES_DIR),
    'FILES_DIR': str(FILES_DIR),
    'LOGS_DIR': str(LOGS_DIR),
    'NOTEBOOK_EXPORTS_DIR': str(NOTEBOOK_EXPORTS_DIR),
    'PHASE1_DIR': str(PHASE1_DIR),
    'PHASE2_DIR': str(PHASE2_DIR),
    'PHASE3_DIR': str(PHASE3_DIR),
    'KAGGLE_RAW_COPY_DIR': str(KAGGLE_RAW_COPY_DIR),
}

paths_json_path = THESIS_ROOT / 'project_paths.json'
with open(paths_json_path, 'w', encoding='utf-8') as f:
    json.dump(project_paths, f, indent=2)

print('Saved project path registry to:', paths_json_path)


Saved project path registry to: /content/drive/MyDrive/NyanThesisAnomaly/project_paths.json


In [ ]:
#Cell 9 - Locate the main outpatient claims file candidate
import pandas as pd

candidate_files = []
for p in KAGGLE_RAW_COPY_DIR.rglob('*'):
    if p.is_file() and p.suffix.lower() in ['.csv', '.txt', '.parquet']:
        candidate_files.append(str(p))

candidates_df = pd.DataFrame({'file_path': sorted(candidate_files)})
display(candidates_df.head(50))

candidates_csv_path = TABLES_DIR / 'dataset_file_inventory.csv'
candidates_df.to_csv(candidates_csv_path, index=False)
print('Saved dataset inventory to:', candidates_csv_path)


,file_path
0,/content/drive/MyDrive/NyanThesisAnomaly/data/...


Saved dataset inventory to: /content/drive/MyDrive/NyanThesisAnomaly/outputs/tables/dataset_file_inventory.csv


In [ ]:
#Cell 10 - Preview the most likely outpatient claims table
preview_df = None
preview_path = None

for p in KAGGLE_RAW_COPY_DIR.rglob('*'):
    if p.is_file() and p.suffix.lower() == '.csv':
        try:
            temp_df = pd.read_csv(p, nrows=5)
            preview_df = temp_df
            preview_path = p
            break
        except Exception:
            continue

print('Preview source file:', preview_path)
if preview_df is not None:
    display(preview_df)
else:
    print('No previewable CSV file found automatically. Review Cell 9 output.')


Preview source file: /content/drive/MyDrive/NyanThesisAnomaly/data/raw/kaggle_cms_desynpuf_2010_outpatient_claims/desynpuf_outpatient_claims_2010_filtered.csv


,DESYNPUF_ID,CLM_ID,SEGMENT,CLM_FROM_DT,CLM_THRU_DT,PRVDR_NUM,CLM_PMT_AMT,NCH_PRMRY_PYR_CLM_PD_AMT,AT_PHYSN_NPI,OP_PHYSN_NPI,...,HCPCS_CD_36,HCPCS_CD_37,HCPCS_CD_38,HCPCS_CD_39,HCPCS_CD_40,HCPCS_CD_41,HCPCS_CD_42,HCPCS_CD_43,HCPCS_CD_44,HCPCS_CD_45
0,00024B3D2352D2D0,542372281246633,1,2010-05-26,2010-05-26,5200YU,40.0,0.0,5.972737e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,0002F28CE057345B,542702280893497,1,2010-04-28,2010-04-28,3900RQ,50.0,0.0,3.981877e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,000345A39D4157C9,542712281107503,1,2010-08-01,2010-08-01,2300CS,90.0,0.0,2.146330e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,000489E7EAAD463F,542912281339207,1,2010-12-12,2010-12-12,1513WQ,300.0,0.0,2.462820e+09,2.462820e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,0007F12A492FD25D,542862281135179,1,2010-01-24,2010-01-24,4200BS,200.0,0.0,4.341916e+09,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
#Cell 11 - Save a lightweight dataset snapshot for quick reuse
if preview_path is not None:
    sample_df = pd.read_csv(preview_path, nrows=1000)
    sample_out_path = DATA_INTERIM_DIR / 'outpatient_claims_sample_1000.csv'
    sample_df.to_csv(sample_out_path, index=False)
    print('Saved sample dataset to:', sample_out_path)
    print('Sample shape:', sample_df.shape)
else:
    print('Skipping sample export because preview file was not found.')


Saved sample dataset to: /content/drive/MyDrive/NyanThesisAnomaly/data/interim/outpatient_claims_sample_1000.csv
Sample shape: (1000, 76)


In [ ]:
#Cell 12 - Export a notebook run summary for later notebooks and reporting
run_summary = {
    'thesis_root': str(THESIS_ROOT),
    'raw_dataset_dir': str(KAGGLE_RAW_COPY_DIR),
    'dataset_inventory_csv': str(TABLES_DIR / 'dataset_file_inventory.csv'),
    'sample_dataset_csv': str(DATA_INTERIM_DIR / 'outpatient_claims_sample_1000.csv'),
    'preview_file': str(preview_path) if preview_path is not None else None,
}

run_summary_path = NOTEBOOK_EXPORTS_DIR / '01_data_setup_run_summary.json'
with open(run_summary_path, 'w', encoding='utf-8') as f:
    json.dump(run_summary, f, indent=2)

print('Saved run summary to:', run_summary_path)
print(json.dumps(run_summary, indent=2))


Saved run summary to: /content/drive/MyDrive/NyanThesisAnomaly/outputs/notebook_exports/01_data_setup_run_summary.json
{
  "thesis_root": "/content/drive/MyDrive/NyanThesisAnomaly",
  "raw_dataset_dir": "/content/drive/MyDrive/NyanThesisAnomaly/data/raw/kaggle_cms_desynpuf_2010_outpatient_claims",
  "dataset_inventory_csv": "/content/drive/MyDrive/NyanThesisAnomaly/outputs/tables/dataset_file_inventory.csv",
  "sample_dataset_csv": "/content/drive/MyDrive/NyanThesisAnomaly/data/interim/outpatient_claims_sample_1000.csv",
  "preview_file": "/content/drive/MyDrive/NyanThesisAnomaly/data/raw/kaggle_cms_desynpuf_2010_outpatient_claims/desynpuf_outpatient_claims_2010_filtered.csv"
}


In [ ]:
df = pd.read_csv("/content/drive/MyDrive/NyanThesisAnomaly/data/raw/kaggle_cms_desynpuf_2010_outpatient_claims/desynpuf_outpatient_claims_2010_filtered.csv")

print(df.shape)
print(df.columns.tolist())
print(df.dtypes)

/tmp/ipykernel_3123/3860067733.py:1: DtypeWarning: Columns (23,24,25,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/content/drive/MyDrive/NyanThesisAnomaly/data/raw/kaggle_cms_desynpuf_2010_outpatient_claims/desynpuf_outpatient_claims_2010_filtered.csv")


(175005, 76)
['DESYNPUF_ID', 'CLM_ID', 'SEGMENT', 'CLM_FROM_DT', 'CLM_THRU_DT', 'PRVDR_NUM', 'CLM_PMT_AMT', 'NCH_PRMRY_PYR_CLM_PD_AMT', 'AT_PHYSN_NPI', 'OP_PHYSN_NPI', 'OT_PHYSN_NPI', 'NCH_BENE_BLOOD_DDCTBL_LBLTY_AM', 'ICD9_DGNS_CD_1', 'ICD9_DGNS_CD_2', 'ICD9_DGNS_CD_3', 'ICD9_DGNS_CD_4', 'ICD9_DGNS_CD_5', 'ICD9_DGNS_CD_6', 'ICD9_DGNS_CD_7', 'ICD9_DGNS_CD_8', 'ICD9_DGNS_CD_9', 'ICD9_DGNS_CD_10', 'ICD9_PRCDR_CD_1', 'ICD9_PRCDR_CD_2', 'ICD9_PRCDR_CD_3', 'ICD9_PRCDR_CD_4', 'ICD9_PRCDR_CD_5', 'ICD9_PRCDR_CD_6', 'NCH_BENE_PTB_DDCTBL_AMT', 'NCH_BENE_PTB_COINSRNC_AMT', 'ADMTNG_ICD9_DGNS_CD', 'HCPCS_CD_1', 'HCPCS_CD_2', 'HCPCS_CD_3', 'HCPCS_CD_4', 'HCPCS_CD_5', 'HCPCS_CD_6', 'HCPCS_CD_7', 'HCPCS_CD_8', 'HCPCS_CD_9', 'HCPCS_CD_10', 'HCPCS_CD_11', 'HCPCS_CD_12', 'HCPCS_CD_13', 'HCPCS_CD_14', 'HCPCS_CD_15', 'HCPCS_CD_16', 'HCPCS_CD_17', 'HCPCS_CD_18', 'HCPCS_CD_19', 'HCPCS_CD_20', 'HCPCS_CD_21', 'HCPCS_CD_22', 'HCPCS_CD_23', 'HCPCS_CD_24', 'HCPCS_CD_25', 'HCPCS_CD_26', 'HCPCS_CD_27', 'HCPCS_CD_28

In [ ]:
df.isna().sum().sort_values(ascending=False).head(30)

,0
HCPCS_CD_45,175005
ICD9_PRCDR_CD_6,175000
ICD9_PRCDR_CD_5,174999
ICD9_PRCDR_CD_4,174995
ICD9_PRCDR_CD_3,174990
ICD9_PRCDR_CD_2,174981
ICD9_PRCDR_CD_1,174960
ICD9_DGNS_CD_10,174291
HCPCS_CD_44,172451
HCPCS_CD_43,172302


## Notebook Run Summary

This notebook sets up the project environment, downloads a raw dataset, organizes the data, and creates initial data snapshots for further analysis.

### Processes Carried Out:

1.  **Project Configuration and Path Setup (Cells 1-3, 8):**
    *   Defined the main thesis root in Google Drive: `/content/drive/MyDrive/NyanThesisAnomaly`.
    *   Mounted Google Drive.
    *   Created a standardized folder structure for raw, interim, and processed data, outputs (figures, tables, files, logs, notebook exports), and phase-specific directories.
    *   Registered all these project paths in a `project_paths.json` file for reuse.

2.  **Library Installation (Cell 4):**
    *   Installed necessary Python libraries: `kagglehub`, `pyarrow`, `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`, `openpyxl`.

3.  **Dataset Acquisition (Cells 5-7):**
    *   Downloaded the 'CMS DESYNPuf 2010 Outpatient Claims' dataset from Kaggle using `kagglehub`.
    *   Inspected the downloaded files, identifying `desynpuf_outpatient_claims_2010_filtered.csv`.
    *   Copied the raw dataset from the Kaggle cache to the project's raw data directory: `/content/drive/MyDrive/NyanThesisAnomaly/data/raw/kaggle_cms_desynpuf_2010_outpatient_claims`.

4.  **Dataset Inventory and Preview (Cells 9-10):**
    *   Scanned the copied raw data directory for CSV, TXT, or Parquet files and created an inventory.
    *   Previewed the first 5 rows of the main CSV file: `desynpuf_outpatient_claims_2010_filtered.csv`.

5.  **Sample Dataset Creation (Cell 11):**
    *   Created a lightweight sample of the main dataset (1000 rows) and saved it to the interim data directory: `/content/drive/MyDrive/NyanThesisAnomaly/data/interim/outpatient_claims_sample_1000.csv`.

### Key Outputs and Artifacts:

*   **Project Path Registry:** `/content/drive/MyDrive/NyanThesisAnomaly/project_paths.json`
*   **Raw Dataset Copy Location:** `/content/drive/MyDrive/NyanThesisAnomaly/data/raw/kaggle_cms_desynpuf_2010_outpatient_claims`
*   **Dataset File Inventory (CSV):** `/content/drive/MyDrive/NyanThesisAnomaly/outputs/tables/dataset_file_inventory.csv`
*   **Sample Dataset (CSV):** `/content/drive/MyDrive/NyanThesisAnomaly/data/interim/outpatient_claims_sample_1000.csv`
*   **Notebook Run Summary (JSON):** `/content/drive/MyDrive/NyanThesisAnomaly/outputs/notebook_exports/01_data_setup_run_summary.json`

### Data Snapshots:

**Dataset File Inventory (`dataset_file_inventory.csv`):**

```
                                          file_path
0  /content/drive/MyDrive/NyanThesisAnomaly/data/raw/kaggle_cms_desynpuf_2010_outpatient_claims/desynpuf_outpatient_claims_2010_filtered.csv
```

**Preview of the Main Outpatient Claims Table (First 5 Rows):**

```
        DESYNPUF_ID           CLM_ID  SEGMENT CLM_FROM_DT CLM_THRU_DT  ...  
0  00024B3D2352D2D0  542372281246633        1  2010-05-26  2010-05-26  ...   
1  0002F28CE057345B  542702280893497        1  2010-04-28  2010-04-28  ...   
2  000345A39D4157C9  542712281107503        1  2010-08-01  2010-08-01  ...   
3  000489E7EAAD463F  542912281339207        1  2010-12-12  2010-12-12  ...   
4  0007F12A492FD25D  542862281135179        1  2010-01-24  2010-01-24  ...   

[5 rows x 76 columns]
```

This notebook has successfully prepared the foundational environment and data for subsequent analysis phases.